<a href="https://colab.research.google.com/github/Gegee-notavailable/thai-fake-news-detection-TFSVM/blob/main/fake_news_bert.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install transformers datasets accelerate torch -q
print("ติดตั้งไลบรารีสำเร็จ!")


ติดตั้งไลบรารีสำเร็จ!


In [16]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# 1. โหลด Dataset และ Tokenizer
dataset = load_dataset("EXt1/Thai-True-Fake-News")
model_name = "airesearch/wangchanberta-base-att-spm-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# 2. ฟังก์ชันแปลงข้อมูล
def preprocess_function(examples):
    result = tokenizer(examples['Title'], truncation=True, padding='max_length', max_length=512)
    labels = [1 if status == 'ข่าวจริง' else 0 for status in examples['Verification_Status']]
    result['labels'] = labels
    return result

# 3. แปลงข้อมูลและเซ็ต Format ให้เป็น PyTorch Tensors
encoded_dataset = dataset.map(preprocess_function, batched=True)
encoded_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

# 4. แบ่งข้อมูล Train / Validation (80:20)
train_testvalid = encoded_dataset['train'].train_test_split(test_size=0.2, seed=42)
train_dataset = train_testvalid['train']
eval_dataset = train_testvalid['test']

# 5. โหลดโมเดลสำหรับจำแนกประเภท (2 คลาส: ข่าวจริง/ข่าวปลอม)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# 6. ตั้งค่าฟังก์ชันวัดผล
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary', zero_division=0)
    acc = accuracy_score(labels, preds)
    return {'accuracy': acc, 'f1': f1, 'precision': precision, 'recall': recall}

# 7. ตั้งค่าการเทรน
training_args = TrainingArguments(
    output_dir='./results',
    eval_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=10,
    save_strategy='epoch',
    load_best_model_at_end=True,
)

# 8. สร้าง Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics,
)

# 9. เริ่มเทรน!
print("เริ่มกระบวนการเทรน Thai BERT...")
trainer.train()

# 10. บันทึกโมเดลและ Tokenizer
model.save_pretrained('./thai_bert_fake_news_model')
tokenizer.save_pretrained('./thai_bert_fake_news_model')
print("เทรนและบันทึกโมเดล Thai BERT สำเร็จเรียบร้อยแล้ว!")

Map:   0%|          | 0/6004 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] CamembertForSequenceClassification LOAD REPORT from: airesearch/wangchanberta-base-att-spm-uncased
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.weight        | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


เริ่มกระบวนการเทรน Thai BERT...


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.386124,0.327805,0.868443,0.866779,0.886207,0.848185
2,0.429751,0.248945,0.904246,0.905350,0.903120,0.907591
3,0.211146,0.285430,0.913405,0.913907,0.916944,0.910891


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

เทรนและบันทึกโมเดล Thai BERT สำเร็จเรียบร้อยแล้ว!


In [27]:
import pandas as pd
from datasets import Dataset

# 1. รายชื่อไฟล์ทั้ง 4 ไฟล์ของคุณ
file_list = [
    "AFNC_Opendata_export_20260930221100(2025) (1).csv",
    "AFNC_Opendata_export_20260930221100(2025).csv",
    "AFNC_Opendata_export_20260930221100(2026) (1).csv",
    "AFNC_Opendata_export_20260930221100(2026).csv"
]

# 2. อ่านไฟล์ทั้งหมดด้วย encoding='utf-8-sig'
df_list = [pd.read_csv(f, encoding='utf-8-sig') for f in file_list]
df_external = pd.concat(df_list, ignore_index=True)

# เลือกเฉพาะคอลัมน์ 'หัวข้อข่าว' และ 'ประเภทข่าว' แล้วลบแถวที่มีค่าว่างออก
df_external = df_external[['หัวข้อข่าว', 'ประเภทข่าว']].dropna()

print(f"รวมข้อมูลสำเร็จ! จำนวนแถวหลังจากทำความสะอาด: {len(df_external)} แถว")
print(df_external.head(2))

# 3. แปลงเป็น Hugging Face Dataset
external_dataset = Dataset.from_pandas(df_external)

# 4. ฟังก์ชัน Tokenize สำหรับข้อมูลภายนอก (ใช้คอลัมน์ 'หัวข้อข่าว' และแปลง 'ประเภทข่าว')
def preprocess_external_function(examples):
    result = tokenizer(examples['หัวข้อข่าว'], truncation=True, padding='max_length', max_length=512)
    # ปรับเงื่อนไขตามข้อมูลจริงในคอลัมน์ 'ประเภทข่าว' (เช่น ถ้าเป็น 'ข่าวจริง' ให้เป็น 1, นอกนั้นเป็น 0)
    labels = [1 if str(status).strip() == 'ข่าวจริง' else 0 for status in examples['ประเภทข่าว']]
    result['labels'] = labels
    return result

encoded_external = external_dataset.map(preprocess_external_function, batched=True)
encoded_external.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

# 5. ประเมินผลโมเดล Thai BERT กับชุดข้อมูลภายนอก
external_results = trainer.predict(encoded_external)

print("\n--- ผลการทดสอบ Thai BERT กับชุดข้อมูลภายนอกทั้ง 4 ไฟล์ ---")
for key, value in external_results.metrics.items():
    print(f"{key}: {value}")

รวมข้อมูลสำเร็จ! จำนวนแถวหลังจากทำความสะอาด: 518 แถว
                                          หัวข้อข่าว ประเภทข่าว
0  ธ.ก.ส. เปิดสินเชื่อด่วนเพื่อการเกษตร ลงทะเบียน...   ข่าวปลอม
1  รับทำใบขับขี่ออนไลน์ถูกกฎหมาย ไม่ต้องสอบเองที่...   ข่าวปลอม


Map:   0%|          | 0/518 [00:00<?, ? examples/s]


--- ผลการทดสอบ Thai BERT กับชุดข้อมูลภายนอกทั้ง 4 ไฟล์ ---
test_loss: 0.42918023467063904
test_accuracy: 0.806949806949807
test_f1: 0.7619047619047619
test_precision: 0.8080808080808081
test_recall: 0.7207207207207207
test_runtime: 15.873
test_samples_per_second: 32.634
test_steps_per_second: 4.095
